# A/B Test Analysis
We're going to conduct an Independent Samples T-test to analyse our A/B test. An Indepedent Samples T-test compares the differences between two means of two different samples.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

Export your results to a .csv file and save it to you github repository. Import your .csv file, inspect it, and clean it where neccesary.

In [ ]:
# Load the two survey exports. The CSV files sit in the same folder as this notebook,
# so I use relative paths to keep it reproducible on any machine.
# encoding is ISO-8859-1 because the Forms export had some non-utf8 characters.
df_A = pd.read_csv("A(Sheet1) (1).csv", encoding='ISO-8859-1')
df_B = pd.read_csv("B(Sheet1) (1).csv", encoding='ISO-8859-1')


# Drop rows where every value is NaN (empty rows from the export).
df_A_clean = df_A.dropna(how='all')
df_B_clean = df_B.dropna(how='all')


In [3]:
# EDA A
df_A_clean.info() # Is your data in the right format?
# df_A.head() # Quick EDA. No? Clean it, you only want the rows and columns containing likert-score data, saved as integers.

# # EDA B
df_B_clean.info() # Is your data in the right format?
# df_B.head() # Quick EDA. No? Clean it, you only want the rows and columns containing likert-score data, saved as integers.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 28 columns):
 #   Column                                                                                   Non-Null Count  Dtype  
---  ------                                                                                   --------------  -----  
 0   Id                                                                                       35 non-null     int64  
 1   Start time                                                                               35 non-null     object 
 2   Completion time                                                                          35 non-null     object 
 3   Email                                                                                    35 non-null     object 
 4   Name                                                                                     35 non-null     object 
 5   Total points                                                      

Now, let's start analysing our gathered data! This block we won't dive into inferential statistics since it can get complex quite fast; we'll do that in Year 2, block A. For now, you just need to know that we need to test whether the data is normally distributed and whether the variances of both samples are equal. Otherwise, our statistical tests would not be valid. What we are going to statistically test is whether there is a statistically significant difference in the mean of a given variable for version A or B. 

In [4]:
df_A.columns

Index(['Id', 'Start time', 'Completion time', 'Email', 'Name', 'Total points',
       'Quiz feedback', 'Grade posted time', 'Points - Column',
       'Feedback - Column', '.I understood what I could use the app for.',
       'Points - .I understood what I could use the app for.',
       'Feedback - .I understood what I could use the app for.',
       '.I found the application intuitive to use.',
       'Points - .I found the application intuitive to use.',
       'Feedback - .I found the application intuitive to use.',
       '.I thought the application was useful.',
       'Points - .I thought the application was useful.',
       'Feedback - .I thought the application was useful.',
       '.I enjoyed using the application.',
       'Points - .I enjoyed using the application.',
       'Feedback - .I enjoyed using the application.',
       '.The app's skin detection feature made me feel more aware of my skin health.',
       'Points - .The app's skin detection feature made me feel more 

In [5]:
questions = [
    ".I enjoyed using the application.",
    ".The app's skin detection feature made me feel more aware of my skin health.",
    ".I found the skin type detection feature helpful and easy to understand.",
    ".I thought the application was useful.",
    ".I found the application intuitive to use.",
    '.I understood what I could use the app for.'
    ]#add the variable names for all your questions of interest

In [6]:
# Step 1: Likert scale mapping
likert_map = {
    "Completely Disagree": 1,
    
    "Neutral": 4,
    "Completely Agree": 7
}

# Step 2: Replace values in both DataFrames
df_A[questions] = df_A[questions].replace(likert_map)
df_B[questions] = df_B[questions].replace(likert_map)


<TEMP>\3071041406.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_A[questions] = df_A[questions].replace(likert_map)
<TEMP>\3071041406.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_B[questions] = df_B[questions].replace(likert_map)


In [7]:
# Replace all NaNs in selected columns with 3 (Neutral)
df_A[questions] = df_A[questions].fillna(3)
df_B[questions] = df_B[questions].fillna(3)


In [8]:
questions = [
    ".I enjoyed using the application.",
    ".The app's skin detection feature made me feel more aware of my skin health.",
    ".I found the skin type detection feature helpful and easy to understand.",
    ".I thought the application was useful.",
    ".I found the application intuitive to use.",
    '.I understood what I could use the app for.'
]

# Convert in both datasets
df_A[questions] = df_A[questions].apply(pd.to_numeric, errors='coerce').astype('Int64')
df_B[questions] = df_B[questions].apply(pd.to_numeric, errors='coerce').astype('Int64')


In [9]:
print(df_A.dtypes)
print(df_B.dtypes)


Id                                                                                           int64
Start time                                                                                  object
Completion time                                                                             object
Email                                                                                       object
Name                                                                                        object
Total points                                                                               float64
Quiz feedback                                                                              float64
Grade posted time                                                                          float64
Points - Column                                                                              int64
Feedback - Column                                                                          float64
.I underst

### Test Assumptions

In [10]:


for q in questions:
    # Run the shapiro-wilk statistical test for each question to check whether the data is normally distributed
    normal_a = stats.shapiro(df_A[q])
    normal_b = stats.shapiro(df_B[q])

    # Check whether the variance of both samples is equal
    homogeneity = stats.levene(df_A[q],
                            df_B[q])

    print(q)
    print(f"This is the p-value version A: ", normal_a)
    print(f"This is the p-value version B: ", normal_b)
    print("A p-value above 0.05 means the data is normally distributed. If for either version A or B the data is not normally distributed, you will have to run the bootstrapped version for this question.")
    print(f"This is the p-value for homogeneity: ", homogeneity)
    print("If the p-value is above 0.05, then the groups have equal variances. If the variance aren't equal then you will have to run the bootstrapped version for this question.")

.I enjoyed using the application.
This is the p-value version A:  ShapiroResult(statistic=0.45459302738004437, pvalue=2.877164535031386e-10)
This is the p-value version B:  ShapiroResult(statistic=0.5593926211392842, pvalue=1.821282929920046e-05)
A p-value above 0.05 means the data is normally distributed. If for either version A or B the data is not normally distributed, you will have to run the bootstrapped version for this question.
This is the p-value for homogeneity:  LeveneResult(statistic=0.8043396932285822, pvalue=0.3747929246081596)
If the p-value is above 0.05, then the groups have equal variances. If the variance aren't equal then you will have to run the bootstrapped version for this question.
.The app's skin detection feature made me feel more aware of my skin health.
This is the p-value version A:  ShapiroResult(statistic=0.6641196128934888, pvalue=1.0796484605432995e-07)
This is the p-value version B:  ShapiroResult(statistic=0.7054702155743977, pvalue=0.0010262157925522

### Run the T-test

Here you will run the T-test. Based on the assumption check above, decide for each question whether you have to run the bootstrapped version or the regular version of the t-test. Run only the appropriate one, and copy the cells as many times as necessary. 

In [11]:
# Run Independent Samples T-test when assumptions are not violated.
results = stats.ttest_ind(df_A[".I enjoyed using the application."],
                          df_B[".I enjoyed using the application."])

# Print the results
print(f"The results are significant if the p-value is significant, which means smaller than 0.05", 
results)

The results are significant if the p-value is significant, which means smaller than 0.05 TtestResult(statistic=-0.8968498721796091, pvalue=0.3747929246081598, df=43.0)


In [12]:
# Run Bootstrapped Independent Samples T-test when assumptions are violated
rng = np.random.default_rng() # create random sampling

results = stats.ttest_ind(df_A[".I found the skin type detection feature helpful and easy to understand."],
                          df_B[".I found the skin type detection feature helpful and easy to understand."],
                          random_state = rng)

# Print the results
print(f"The results are significant if the p-value is significant, which means smaller than 0.05", 
results)

The results are significant if the p-value is significant, which means smaller than 0.05 TtestResult(statistic=-0.7275477052618896, pvalue=0.47083356970969437, df=43.0)


<TEMP>\1939203647.py:4: DeprecationWarning: Arguments {'random_state'} are deprecated, whether passed by position or keyword. They will be removed in SciPy 1.17.0. Use ``method`` to perform a permutation test.
  results = stats.ttest_ind(df_A[".I found the skin type detection feature helpful and easy to understand."],


In [13]:
# Run Bootstrapped Independent Samples T-test when assumptions are violated
rng = np.random.default_rng() # create random sampling

results = stats.ttest_ind(df_A[".I thought the application was useful."],
                          df_B[".I thought the application was useful."],
                          random_state = rng)

# Print the results
print(f"The results are significant if the p-value is significant, which means smaller than 0.05", 
results)

The results are significant if the p-value is significant, which means smaller than 0.05 TtestResult(statistic=-0.20858472746796117, pvalue=0.8357573566331512, df=43.0)


<TEMP>\3936555644.py:4: DeprecationWarning: Arguments {'random_state'} are deprecated, whether passed by position or keyword. They will be removed in SciPy 1.17.0. Use ``method`` to perform a permutation test.
  results = stats.ttest_ind(df_A[".I thought the application was useful."],


In [14]:
# Run Bootstrapped Independent Samples T-test when assumptions are violated
rng = np.random.default_rng() # create random sampling

results = stats.ttest_ind(df_A['.I understood what I could use the app for.'],
                          df_B['.I understood what I could use the app for.'],
                          random_state = rng)

# Print the results
print(f"The results are significant if the p-value is significant, which means smaller than 0.05", 
results)

The results are significant if the p-value is significant, which means smaller than 0.05 TtestResult(statistic=-0.6166464696428456, pvalue=0.5407192097065592, df=43.0)


<TEMP>\3276589658.py:4: DeprecationWarning: Arguments {'random_state'} are deprecated, whether passed by position or keyword. They will be removed in SciPy 1.17.0. Use ``method`` to perform a permutation test.
  results = stats.ttest_ind(df_A['.I understood what I could use the app for.'],


If the results are significant, that means that the means between the two versions of this specific question are different enough to exclude chance for being the cause. So if the B version has a higher/lower average score and is statistically significant, then it works better/worse. If the results are not significant then the changes don't have a real measurable effect so maybe it's not better or maybe the questions don't really measure the effect. There's more to it but we will leave it up to inferential statistics in year 2.

In [20]:
# Descriptive stats for one question in Group A
a = df_A[".I found the skin type detection feature helpful and easy to understand."].describe()

b = df_B[".I found the skin type detection feature helpful and easy to understand."].describe()

print(a)
print(b)


count        35.0
mean     3.685714
std      1.409452
min           3.0
25%           3.0
50%           3.0
75%           3.5
max           7.0
Name: .I found the skin type detection feature helpful and easy to understand., dtype: Float64
count       10.0
mean         4.1
std      2.13177
min          1.0
25%          3.0
50%          3.0
75%         6.25
max          7.0
Name: .I found the skin type detection feature helpful and easy to understand., dtype: Float64


In [25]:
for i in questions:
    a=df_A[i].describe()
    b=df_B[i].describe()
    print({i})
    print("Group A:")
    print(a)
    print("Group B:")
    print(b)
    print("-" * 50)    

{'.I enjoyed using the application.'}
Group A:
count        35.0
mean     3.314286
std      0.758149
min           3.0
25%           3.0
50%           3.0
75%           3.0
max           7.0
Name: .I enjoyed using the application., dtype: Float64
Group B:
count        10.0
mean          3.6
std      1.264911
min           3.0
25%           3.0
50%           3.0
75%          3.75
max           7.0
Name: .I enjoyed using the application., dtype: Float64
--------------------------------------------------
{".The app's skin detection feature made me feel more aware of my skin health."}
Group A:
count        35.0
mean          3.2
std      1.389033
min           1.0
25%           3.0
50%           3.0
75%           3.0
max           7.0
Name: .The app's skin detection feature made me feel more aware of my skin health., dtype: Float64
Group B:
count        10.0
mean          4.4
std      1.837873
min           3.0
25%           3.0
50%           3.5
75%          6.25
max           7.0
Name: .